# Climate Claim Scanner — Daily GPU Maintenance

**In-house maintenance pass. Run 2–3×/week (or daily).** Local ingestion runs
autonomously every 24h on CPU and accumulates raw posts; this notebook does the
GPU-heavy stages and exports results back for the website:

`classify → vision → reindex → evaluate` (see `climate_verifier/maintenance.py`).

Each stage writes a health heartbeat (`data/health.json`) and evaluation appends a
drift snapshot (`data/eval_history.jsonl`) the app plots. Set `DRIVE_DIR` below to
your Google Drive sync folder.


## 1 · Confirm GPU


In [ ]:
!nvidia-smi -L

## 2 · Clone / update the repo (public — no token needed)


In [ ]:
import os
REPO   = "https://github.com/Sadaf-Burhan/ClimateClaimVerifier.git"
BRANCH = "multimodal-edge-gating"
if not os.path.exists("ClimateClaimVerifier"):
    !git clone -b {BRANCH} {REPO}
%cd ClimateClaimVerifier
!git pull origin {BRANCH}

## 3 · Install the package + dependencies


In [ ]:
!pip install -e . -q
print("installed")

## 4 · Install Ollama + pull the models
`qwen2.5:3b` = classifier, `qwen2.5vl:7b` = vision (GPU-only).


In [ ]:
# zstd MUST be installed FIRST — Colab's image doesn't ship it and Ollama's installer needs it to
# unpack, so the install fails without this line. Then serve in the background and give the server
# a moment to come up before pulling models.
#
# VERSION PINNING: Colab re-installs Ollama and re-pulls the model on EVERY run, both unpinned — so
# the runtime and the model build can change underneath you between runs. That is invisible in the
# results and would be misread as classifier drift on the frozen gold set. Set OLLAMA_VERSION below
# to a known-good version (read it off the drift log's `ollama_version` after a good run) to freeze
# the runtime; leave it "" to track latest. The model digest is printed + logged either way.
OLLAMA_VERSION = ""      # e.g. "0.5.7" to pin; "" = latest

import os, subprocess, time
!sudo apt-get update && sudo apt-get install -y zstd
if OLLAMA_VERSION:
    os.environ["OLLAMA_VERSION"] = OLLAMA_VERSION
    print(f"pinning Ollama runtime to {OLLAMA_VERSION}")
!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull qwen2.5:3b
!ollama pull qwen2.5vl:7b

# Record exactly what we ended up with — this is what makes a silent model swap visible.
print("\n=== environment fingerprint (compare against the drift log when gold moves) ===")
!ollama --version
!ollama list

## 5 · Pull state + history from Drive
The local daily job writes `data/ingested.db`; sync it here so we classify the freshest posts.
**Edit `DRIVE_DIR`** to your synced folder.

We pull three files, and the distinction matters:
- **`ingested.db`** — *state*. Replaced each run (a dated backup is kept on export).
- **`eval_history.jsonl`** — an *append-only log*. Must be pulled first or the eval writes a
  one-entry file and the export destroys your drift trend.
- **`health.json`** — per-stage heartbeats, merged. Pull it or the Colab run wipes the local
  ingestion/refresh heartbeats.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, shutil
DRIVE_DIR = "/content/drive/MyDrive/ClimateScanner"   # <-- your Drive folder
os.makedirs("data", exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)

# Pull STATE + the ACCUMULATING artifacts. All three are git-ignored, so a fresh clone starts
# EMPTY for them — and §7 exports them back. Without pulling first, this run would write a
# one-entry drift log and a health file missing the local ingestion heartbeat, then overwrite
# the Drive copies with those — destroying the history. Pull-then-append is what makes the drift
# chart a real trend:
#   ingested.db        = state (replaced each run)
#   eval_history.jsonl = append-only LOG (evaluate appends one line per run)
#   health.json        = per-stage heartbeats (update_health merges; preserves ingestion/refresh)
for f in ["ingested.db", "eval_history.jsonl", "health.json"]:
    src = os.path.join(DRIVE_DIR, f)
    if os.path.exists(src):
        shutil.copy(src, os.path.join("data", f))
        print(f"pulled {f}")
    else:
        print(f"no {f} in Drive yet — starting fresh for it")

## 6 · Run the maintenance chain
`--all` = classify → vision → reindex → evaluate. One broken stage doesn't block the
rest; failures are recorded in `data/health.json` and the cell exits non-zero.


In [ ]:
!python -m climate_verifier.maintenance --all

## 7 · Export results back to Drive (for the website)


In [ ]:
import os, shutil, glob, time
from datetime import datetime, timezone

for f in ["ingested.db", "eval_history.jsonl", "health.json"]:
    p = os.path.join("data", f)
    if os.path.exists(p):
        shutil.copy(p, os.path.join(DRIVE_DIR, f))
        print("exported", f)
if os.path.exists("data/chroma_evidence"):
    shutil.copytree("data/chroma_evidence", os.path.join(DRIVE_DIR, "chroma_evidence"), dirs_exist_ok=True)
    print("exported chroma_evidence/")

# Versioned DB backup. `ingested.db` is STATE — replaced every run — so keep a dated copy to roll
# back to if a run corrupts it, and prune past RETAIN_DAYS. NOTE the asymmetry: the drift log
# (eval_history.jsonl) is never versioned OR pruned — it's the history itself, ~250 bytes/run
# (a year of daily runs is ~90KB), and pruning it would throw away the long-range trend.
RETAIN_DAYS = 30
bak_dir = os.path.join(DRIVE_DIR, "backups")
os.makedirs(bak_dir, exist_ok=True)
if os.path.exists("data/ingested.db"):
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    shutil.copy("data/ingested.db", os.path.join(bak_dir, f"ingested_{stamp}.db"))
    print(f"backed up backups/ingested_{stamp}.db")
cutoff = time.time() - RETAIN_DAYS * 86400
for old in glob.glob(os.path.join(bak_dir, "ingested_*.db")):
    if os.path.getmtime(old) < cutoff:
        os.remove(old)
        print("pruned", os.path.basename(old))

print("\nDone. Pull these into the project dir; the app reads DB + eval_history + health.")

## 7b · Week-7 image-extraction eval (multimodal)

Scores the **image-input path** (`extract_from_image` → adapter → classify + region-aware RAG) on
the hand-labeled set in `data/image_eval/labels.jsonl`. It runs **after** the maintenance chain so
the evidence index is fresh (enables the `news_status` agreement — the primary bar).

The screenshot images are git-ignored, so pull them from Drive first: put your local
`data/image_eval/images/*.png` into **`<DRIVE_DIR>/image_eval_images/`**. Non-post rows (no
engagement) are scored as *rejection* cases. Writes `data/image_eval/eval_results.json` and exports
it to Drive as `image_eval_results.json` for download.

In [ ]:
# Pull the labeled screenshots from Drive (they're git-ignored, so NOT in the clone).
# Put your local `data/image_eval/images/*.png` into `<DRIVE_DIR>/image_eval_images/` first.
import os, shutil
_img_src = os.path.join(DRIVE_DIR, "image_eval_images")
_img_dst = "data/image_eval/images"
os.makedirs(_img_dst, exist_ok=True)
if os.path.isdir(_img_src):
    _n = 0
    for f in os.listdir(_img_src):
        shutil.copy(os.path.join(_img_src, f), _img_dst); _n += 1
    print(f"pulled {_n} image(s) from {_img_src}")
else:
    print(f"⚠️ No {_img_src} on Drive — put your screenshots there to score the image eval. "
          "(Labels still load; rows without an image file are skipped.)")
print("images present:", sorted(os.listdir(_img_dst)))

In [ ]:
# Run the image-extraction eval. It runs AFTER `maintenance --all`, so the evidence index is
# fresh and this includes the news_status agreement (the primary bar). Writes eval_results.json,
# then exports it to Drive so you can download the scorecard.
!python scripts/eval_image_extraction.py

import os, shutil
_res = "data/image_eval/eval_results.json"
if os.path.exists(_res):
    shutil.copy(_res, os.path.join(DRIVE_DIR, "image_eval_results.json"))
    print("\n✅ exported image_eval_results.json to", DRIVE_DIR)
else:
    print("\nno eval_results.json produced (no images were scored).")

## 8 · Quick look — latest drift snapshot


In [ ]:
import json, pathlib
p = pathlib.Path("data/eval_history.jsonl")
if p.exists():
    print("latest eval:", json.loads(p.read_text().strip().splitlines()[-1]))
else:
    print("no eval history yet")